# Twitter Prediction Scorer — Colab Launcher

This notebook is a **thin launcher only** — all real logic lives in `src/main.py`.
When you push code changes, just re-run **Step 0** then **Step 4** — no edits needed here.

---
### Before you start — add Colab Secrets (one-time)
Go to the 🔑 **Secrets** tab (left sidebar) and add:

| Secret name | Value |
|---|---|
| `GITHUB_REPO` | `https://github.com/your-username/your-repo.git` |
| `MONGO_URI` | `mongodb://78.138.0.218:27017` |
| `MONGO_DB` | your database name |
| `MONGO_COLLECTION` | your collection name |
| `OPENAI_API_KEY` | your OpenAI key |

Toggle **"Notebook access"** ON for each secret.

## Step 0 — Clone repo (or pull latest after a git push)
Re-run this cell any time you push changes to GitHub.

In [ ]:
import os, sys

REPO_DIR = "/content/repo"

try:
    from google.colab import userdata
    GITHUB_REPO = userdata.get('GITHUB_REPO')
except Exception:
    GITHUB_REPO = os.environ.get('GITHUB_REPO', '')

if not GITHUB_REPO:
    raise ValueError("Add GITHUB_REPO to Colab Secrets.")

if not os.path.exists(REPO_DIR):
    print(f"Cloning {GITHUB_REPO} ...")
    assert os.system(f"git clone {GITHUB_REPO} {REPO_DIR}") == 0, "git clone failed"
    print("✅ Cloned.")
else:
    print("Pulling latest changes ...")
    os.system(f"git -C {REPO_DIR} pull origin main")
    print("✅ Up to date.")

# src/main.py imports `from lib.utlis import timeasit`
# %run adds the script's own directory (src/) to sys.path automatically,
# so lib/ resolves correctly without any extra sys.path manipulation.
print(f"Script to run: {REPO_DIR}/src/main.py")

## Step 1 — Install dependencies

In [ ]:
%pip install -q \
    pymongo==4.17.0 \
    yfinance==1.3.0 \
    openai==2.32.0 \
    pandas==3.0.2 \
    numpy==2.4.4 \
    pytz==2026.1.post1 \
    python-dotenv==1.2.2

print('✅ All packages installed.')

## Step 2 — Load secrets into environment variables

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ['MONGO_URI']        = userdata.get('MONGO_URI')
    os.environ['MONGO_DB']         = userdata.get('MONGO_DB')
    os.environ['MONGO_COLLECTION'] = userdata.get('MONGO_COLLECTION')
    os.environ['OPENAI_API_KEY']   = userdata.get('OPENAI_API_KEY')
    print('✅ Secrets loaded from Colab Secrets tab.')
except ModuleNotFoundError:
    from dotenv import load_dotenv
    load_dotenv()
    print('✅ Secrets loaded from .env file.')

required = ['MONGO_URI', 'MONGO_DB', 'MONGO_COLLECTION', 'OPENAI_API_KEY']
missing  = [k for k in required if not os.environ.get(k)]
if missing:
    print(f'⚠️  Missing secrets: {missing}')
else:
    print('✅ All required secrets present.')

## Step 3 — Verify MongoDB connection

In [ ]:
import os
from pymongo import MongoClient
from pymongo.errors import ConnectionFailure

try:
    client = MongoClient(os.environ['MONGO_URI'], serverSelectionTimeoutMS=5000)
    client.admin.command('ping')
    col   = client[os.environ['MONGO_DB']][os.environ['MONGO_COLLECTION']]
    count = col.count_documents({'scored': False})
    print(f'✅ MongoDB connected — {count} unscored record(s) waiting.')
except ConnectionFailure as e:
    print(f'❌ Cannot reach MongoDB: {e}')

## Step 4 — Run the scorer

Executes `src/main.py` from the cloned repo directly.
Any `git push` → re-run Step 0 → re-run this cell. No edits to this notebook needed.

> This cell runs forever. Stop it with the ⏹ button or `Kernel → Interrupt`.

In [ ]:
%run /content/repo/src/main.py

---
## Optional — Demo mode (no real DB or API needed)

Runs `src/demo.py` from the cloned repo.
Same log format and 8-step pipeline — uses fake records and simulated delays.

In [ ]:
%run /content/repo/src/demo.py